In [2]:
import wbgapi as wb
import pandas as pd
import matplotlib.pyplot as plt

# 1.获取所有数据库的原始数据列表

In [3]:
# 1. 获取所有数据库的原始数据列表
sources_raw = wb.source.list()

# 2. 转换为 DataFrame
df_sources = pd.DataFrame(sources_raw)

# 3. 整理列名并查看
df_sources.columns = ['id', 'lastupdated', 'name', 'code', 'databid', 'description', 'url', 'dataavailability', 'metadataavailability', 'concepts'] # 视版本而定，通常有这几项

In [7]:
df_sources

,id,lastupdated,name,code,databid,description,url,dataavailability,metadataavailability,concepts
0,1,2021-08-18,Doing Business,DBS,3001,,,Y,Y,3
1,2,2026-04-08,World Development Indicators,WDI,2,,,Y,Y,3
2,3,2026-03-18,Worldwide Governance Indicators,WGI,1181,,,Y,Y,3
3,5,2016-03-21,Subnational Malnutrition Database,SNM,1267,,,Y,Y,3
4,6,2025-12-03,International Debt Statistics,IDS,14,,,Y,Y,4
...,...,...,...,...,...,...,...,...,...,...
66,89,2025-10-27,Identification for Development (ID4D) Data,ID4,3738,,,Y,Y,3
67,90,2024-08-04,ICP 2021,IC2,3869,,,Y,Y,4
68,91,2026-01-21,PEFA_CRPFM,CRP,3871,,,Y,Y,4
69,92,2025-03-28,Disability Data Hub (DDH),DDH,3872,,,Y,Y,3


In [5]:
df_sources.to_excel('data/worldBank_tabel.xlsx', index=False)

# 主要分析id = 2 World Development Indicators

In [31]:
# 1. 获取原始列表
topics_raw = wb.topic.list()

# 2. 转换为 DataFrame（自动带入原始列名）
df_topics = pd.DataFrame(topics_raw)

# 3.世界银行所有的主题
df_topics

,id,value,sourceNote
0,1,Agriculture & Rural Development,For the 70 percent of the world's poor who liv...
1,2,Aid Effectiveness,Aid effectiveness is the impact that aid has i...
2,3,Economy & Growth,Economic growth is central to economic develop...
3,4,Education,Education is one of the most powerful instrume...
4,5,Energy & Mining,The world economy needs ever-increasing amount...
5,6,Environment,Natural and man-made environmental resources –...
6,7,Financial Sector,An economy's financial markets are critical to...
7,8,Health,Improving health is central to the Millennium ...
8,9,Infrastructure,Infrastructure helps determine the success of ...
9,10,Social Protection & Labor,The supply of labor available in an economy in...


In [39]:
import wbgapi as wb
import pandas as pd

# 获取 WDI (db=2) 的所有指标描述
all_indicators = wb.series.list(db=2)
df_menu = pd.DataFrame(all_indicators)
df_menu.columns = ['指标代码', '指标详细全称']

# 导出到 Excel，你可以像翻阅目录一样慢慢看
df_menu.to_excel("data/WDI_Full_Menu.xlsx", index=False)
print(f"全量目录已生成！总共包含 {len(df_menu)} 个指标。")

全量目录已生成！总共包含 1486 个指标。


In [42]:
# 重新提取逻辑：如果是下划线风格，取前两个部分；如果是点号风格，取第一部分
def clean_prefix(code):
    if '_' in code:
        # 比如 HD_HCIP_EDUC_FE -> HD_HCIP
        return "_".join(code.split('_')[:2])
    else:
        # 比如 NY.GDP.MKTP -> NY
        return code.split('.')[0]

df_menu['归一化领域'] = df_menu['指标代码'].apply(clean_prefix)
df_menu['归一化领域'].value_counts()

归一化领域
SE               156
SH               148
SL               129
SP                98
NY                94
DT                84
IC                76
NE                65
EN                63
GC                45
TM                45
DC                38
NV                35
TX                30
AG                27
EG                27
IQ                27
SI                26
BX                21
ER                15
BM                14
SM                13
BN                12
HD_HCIP           12
SG                12
IT                11
FM                10
SH_UHC            10
ST                10
FX                 9
LP                 9
IE                 8
IS                 8
MS                 8
per_lm             8
per_sa             8
per_si             8
CM                 7
IP                 7
FB                 6
PA                 6
VC                 6
FR                 5
SN                 5
FI                 4
FS                 4
FP                 3
GD_WBL 

## 1. 经济增长与实力 (核心前缀: NY, NE)

In [46]:
def get_economic_comparison(indicators, countries=['CHN', 'JPN'], time_range=range(2000, 2024)):
    """
    通用方法：抓取指定国家和指标的数据并返回清洗后的 DataFrame
    
    :param indicators: 字典格式 {代码: 自定义名称}
    :param countries: 国家代码列表，如 ['CHN', 'JPN']
    :param time_range: 时间范围
    :return: 格式化后的 DataFrame
    """
    # 1. 抓取数据
    # mr=1 表示只取最近的数据，但我们要看趋势，所以指定 time
    df = wb.data.DataFrame(list(indicators.keys()), countries, time=time_range)
    
    # 2. 数据转换 (将多索引展开，方便阅读)
    df.index.names = ['Country', 'Series']
    df = df.stack().unstack('Series').reset_index()
    df.rename(columns={'level_1': 'Year'}, inplace=True)
    
    # 3. 映射指标名称，让结果更易懂
    df.rename(columns=indicators, inplace=True)
    
    # 4. 清理年份格式 (将 Y2020 转为 2020)
    df['Year'] = df['Year'].str.replace('YR', '').astype(int)
    
    return df

# --- 执行分析 ---

# 定义你刚才挑选的“明星指标”
my_indicators = {
    'NY.GDP.MKTP.CD': 'GDP总额',
    'NY.GDP.MKTP.KD.ZG': 'GDP增长率',
    'NY.GDP.PCAP.PP.CD': '人均GDP(PPP)',
    'NE.CON.TOTL.ZS': '消费占比',
    'NE.GDI.FTOT.ZS': '投资占比',
    'NV.IND.TOTL.ZS': '工业占比'
}

# 调用方法
comparison_df = get_economic_comparison(my_indicators)

# 打印最后几行看看效果
comparison_df

Series,Country,Year,消费占比,投资占比,工业占比,GDP总额,GDP增长率,人均GDP(PPP)
0,CHN,2000,63.734767,32.647179,45.073832,1.223755e+12,8.586432,2963.532386
1,CHN,2001,62.134163,33.459882,44.276544,1.355037e+12,8.312639,3258.402107
2,CHN,2002,61.207475,34.987864,43.875803,1.489822e+12,9.243393,3590.765959
3,CHN,2003,58.268455,38.084681,44.982762,1.683903e+12,10.118193,4007.116046
4,CHN,2004,55.523784,39.350537,45.232833,1.984197e+12,10.131407,4504.929372
5,CHN,2005,54.793044,39.270329,46.381622,2.317551e+12,11.458238,5148.182739
6,CHN,2006,53.033718,38.521573,46.886503,2.791498e+12,12.674675,5946.286828
7,CHN,2007,51.461906,37.670987,46.185236,3.604056e+12,14.149985,6935.265806
8,CHN,2008,50.271092,38.803081,46.236419,4.667346e+12,9.670246,7712.846084
9,CHN,2009,50.572067,43.409767,45.178859,5.189577e+12,9.402700,8447.982815


## 2. 人力资本与社会福利 (核心前缀: SP, SH, SE, HD)

## 3. 环境与资源可持续性 (核心前缀: EN, EG, AG)

## 4. 商业、法律与治理 (核心前缀: IC, IQ, GD)